# Hyperparameter Tuning with Optuna

In this notebook, we use Optuna to tune an XGBoost model for predicting hospital readmission using multiclass classification (0 = No, 1 = <30 days, 2 = >30 days).

In [1]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import pandas as pd
from pathlib import Path

In [ ]:
import sys
sys.path.append("../src")
from preprocessing import load_and_preprocess_data

from preprocessing import load_and_preprocess_data


In [ ]:
import optuna

def objective(trial):
    from xgboost import XGBClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import f1_score

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "objective": "multi:softmax",
        "num_class": 3,
        "n_jobs": -1,
        "verbosity": 0
    }

    model = XGBClassifier(**param)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    return f1